In [25]:
# ── Cell 0: Environment setup ─────────────────────────────────────────────────
import os, torch, shutil
from pathlib import Path

# download_data.py places EK100_MIR/ next to src/ by default.
# Override with: export EK100_MIR_ROOT=/your/path
_here = Path(os.getcwd())
EK100_MIR_ROOT = Path(os.environ.get("EK100_MIR_ROOT", _here.parent / "EK100_MIR"))

assert EK100_MIR_ROOT.exists(), (
    f"EK100_MIR not found at {EK100_MIR_ROOT}\n"
    "Run  python src/download_data.py  first, or set EK100_MIR_ROOT env var."
)

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU : {gpu.name}  ({gpu.total_memory/1e9:.1f} GB VRAM)")
    print(f"CUDA: {torch.version.cuda} | PyTorch: {torch.__version__}")
else:
    print("No GPU — inference will run on CPU (may be slow)")

_, _, free = shutil.disk_usage(EK100_MIR_ROOT)
print(f"Disk free : {free/1e9:.0f} GB")
print(f"EK100_MIR : {EK100_MIR_ROOT.resolve()}")

GPU : NVIDIA GeForce RTX 5070  (12.3 GB VRAM)
CUDA: 13.0 | PyTorch: 2.11.0+cu130
Disk free : 1408 GB
EK100_MIR : /home/nico/Desktop/Multi-Instance-Retrieval-EK-100/EK100_MIR


In [26]:
# ── Cell 1: Clone repos ──────────────────────────────────────────────────────
# MI-MM and JPoSE provide different model architectures (S3D + sentence
# embeddings vs. TBN + part-of-speech embeddings). We clone both so we can
# run all three models (MI-MM, JPoSE, MLP/MMEN) and ensemble their outputs.
import subprocess

REPOS_DIR = EK100_MIR_ROOT / "repos"
REPOS_DIR.mkdir(exist_ok=True)

repos = {
    "MI-MM":                           "https://github.com/adrianofragomeni/MI-MM.git",
    "Joint-Part-of-Speech-Embeddings": "https://github.com/mwray/Joint-Part-of-Speech-Embeddings.git",
}

for name, url in repos.items():
    dest = REPOS_DIR / name
    if not dest.exists():
        r = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", url, str(dest)],
            capture_output=True, text=True,
        )
        if r.returncode != 0:
            raise RuntimeError(f"Clone failed for {name}:\n{r.stderr}")
        print(f"Cloned : {dest}")
    else:
        print(f"Exists : {dest}")

Exists : /home/nico/Desktop/Multi-Instance-Retrieval-EK-100/EK100_MIR/repos/MI-MM
Exists : /home/nico/Desktop/Multi-Instance-Retrieval-EK-100/EK100_MIR/repos/Joint-Part-of-Speech-Embeddings


In [27]:
# ── Cell 2: Configure paths ──────────────────────────────────────────────────
# Both repos ship an empty data/ directory with .gitkeep placeholders so git
# tracks the folder structure. We remove that placeholder and replace it with
# a symlink to the actual data downloaded by download_data.py. This preserves
# the relative path assumptions hardcoded in each model's source files.
import re, pickle, shutil
import numpy as np
from pathlib import Path

MIMM_DIR  = EK100_MIR_ROOT / "repos" / "MI-MM"
JPOSE_DIR = EK100_MIR_ROOT / "repos" / "Joint-Part-of-Speech-Embeddings"

def link_data(repo_dir, data_target):
    """Replace a git placeholder data/ dir with a symlink to data_target."""
    link = repo_dir / "data"
    if link.is_symlink():
        print(f"  symlink exists : {repo_dir.name}/data")
    elif link.is_dir():
        shutil.rmtree(link)
        link.symlink_to(data_target)
        print(f"  linked         : {repo_dir.name}/data -> {data_target}")
    else:
        link.symlink_to(data_target)
        print(f"  linked         : {repo_dir.name}/data -> {data_target}")

print("Setting up data symlinks:")
link_data(MIMM_DIR,  EK100_MIR_ROOT / "data" / "MI-MM")
link_data(JPOSE_DIR, EK100_MIR_ROOT / "data" / "JPoSE" / "data")

# MI-MM: testing.py runs from MI-MM/src/ so all paths are ../data/...
MIMM_OUTPUT_DIR = MIMM_DIR / "output"
MIMM_OUTPUT_DIR.mkdir(exist_ok=True)

# JPoSE / MLP
MODELS_DIR     = JPOSE_DIR / "data" / "models"
VID_FEAT_DIR   = JPOSE_DIR / "data" / "video_features"
TXT_FEAT_DIR   = JPOSE_DIR / "data" / "text_features"
DATAFRAMES_DIR = JPOSE_DIR / "data" / "dataframes"
RELATIONAL_DIR = JPOSE_DIR / "data" / "relational"
RELEVANCY_DIR  = JPOSE_DIR / "data" / "relevancy"

SUBMISSIONS_DIR = EK100_MIR_ROOT / "submissions"
ZIPS_DIR        = EK100_MIR_ROOT / "submission_zips"
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
ZIPS_DIR.mkdir(parents=True, exist_ok=True)

print("\nPaths:")
print(f"  MI-MM  repo  : {MIMM_DIR}")
print(f"  JPoSE  repo  : {JPOSE_DIR}")
print(f"  Submissions  : {SUBMISSIONS_DIR}")
print(f"  ZIPs         : {ZIPS_DIR}")

Setting up data symlinks:
  linked         : MI-MM/data -> /home/nico/Desktop/Multi-Instance-Retrieval-EK-100/EK100_MIR/data/MI-MM
  symlink exists : Joint-Part-of-Speech-Embeddings/data

Paths:
  MI-MM  repo  : /home/nico/Desktop/Multi-Instance-Retrieval-EK-100/EK100_MIR/repos/MI-MM
  JPoSE  repo  : /home/nico/Desktop/Multi-Instance-Retrieval-EK-100/EK100_MIR/repos/Joint-Part-of-Speech-Embeddings
  Submissions  : /home/nico/Desktop/Multi-Instance-Retrieval-EK-100/EK100_MIR/submissions
  ZIPs         : /home/nico/Desktop/Multi-Instance-Retrieval-EK-100/EK100_MIR/submission_zips


In [28]:
# ── Cell 3: Verify data ──────────────────────────────────────────────────────
def check_path(path, label, required=True):
    p = Path(path)
    if p.is_dir():
        files  = [f for f in p.rglob("*") if f.is_file()]
        ok     = bool(files)
        detail = f"{len(files)} file(s)  ({sum(f.stat().st_size for f in files)/1e6:.0f} MB)"
    else:
        ok     = p.exists()
        detail = f"{p.stat().st_size/1e6:.0f} MB" if ok else "NOT FOUND"
    icon = "OK" if ok else ("ERR" if required else "WARN")
    print(f"  [{icon}]  {label}: {detail}")
    return ok

print("── MI-MM ────────────────────────────────────────────────")
mimm_ok = all([
    check_path(MIMM_DIR / "data" / "dataframes", "dataframes"),
    check_path(MIMM_DIR / "data" / "features",   "S3D features"),
    check_path(MIMM_DIR / "data" / "models",     "models"),
    check_path(MIMM_DIR / "data" / "relevancy",  "relevancy"),
    check_path(MIMM_DIR / "data" / "resources",  "resources"),
])

print("\n── JPoSE / MLP ──────────────────────────────────────────")
jpose_ok = all([
    check_path(MODELS_DIR,     "models"),
    check_path(VID_FEAT_DIR,   "video_features"),
    check_path(TXT_FEAT_DIR,   "text_features"),
    check_path(DATAFRAMES_DIR, "dataframes"),
    check_path(RELATIONAL_DIR, "relational"),
    check_path(RELEVANCY_DIR,  "relevancy"),
])
check_path(MODELS_DIR / "JPoSE_BEST" / "model" / "EPIC_100_retrieval_JPoSE_BEST.pth", "JPoSE checkpoint")
check_path(MODELS_DIR / "MMEN_BEST"  / "model" / "EPIC_100_retrieval_MLP_BEST.pth",   "MLP  checkpoint")

if not (mimm_ok and jpose_ok):
    raise RuntimeError("Missing data — run: python src/download_data.py")
print("\nAll data OK")

── MI-MM ────────────────────────────────────────────────
  [OK]  dataframes: 5 file(s)  (12 MB)
  [OK]  S3D features: 2 file(s)  (315 MB)
  [OK]  models: 1 file(s)  (154 MB)
  [OK]  relevancy: 1 file(s)  (8598 MB)
  [OK]  resources: 4 file(s)  (131 MB)

── JPoSE / MLP ──────────────────────────────────────────
  [OK]  models: 5 file(s)  (12 MB)
  [OK]  video_features: 6 file(s)  (1955 MB)
  [OK]  text_features: 10 file(s)  (107 MB)
  [OK]  dataframes: 7 file(s)  (20 MB)
  [OK]  relational: 4 file(s)  (3 MB)
  [OK]  relevancy: 2 file(s)  (93 MB)
  [OK]  JPoSE checkpoint: 8 MB
  [OK]  MLP  checkpoint: 4 MB

All data OK


In [29]:
# ── Cell 4: Apply compatibility patches ──────────────────────────────────────
# pickle5 was folded into the stdlib in Python 3.8 (it's now just `pickle`).
# torch.load changed its default for weights_only in PyTorch >= 2.6: it must
# now be set explicitly to False to deserialise full checkpoints that contain
# custom Python objects. patch_file is idempotent — safe to re-run.

def patch_file(path, old, new):
    p = Path(path)
    if not p.exists():
        return
    content = p.read_text()
    if old in content:
        p.write_text(content.replace(old, new))
        print(f"  patched : {p.name}")

# MI-MM: pickle5 -> pickle
patch_file(
    MIMM_DIR / "src" / "loader" / "loader_features.py",
    "import pickle5 as pickle",
    "import pickle",
)
# MI-MM: torch.load weights_only
patch_file(
    MIMM_DIR / "src" / "testing.py",
    "best_model = th.load(Path(args_.path_model)/args_.best_model)",
    "best_model = th.load(Path(args_.path_model)/args_.best_model, weights_only=False)",
)
patch_file(
    MIMM_DIR / "src" / "models" / "embedding_projection.py",
    'pretrained_dict = th.load(self.path_resources / "s3d_howto100m.pth")',
    'pretrained_dict = th.load(self.path_resources / "s3d_howto100m.pth", weights_only=False)',
)

# JPoSE / MLP: torch.load weights_only (all Python files under src/)
for fpath in (JPOSE_DIR / "src").rglob("*.py"):
    txt = fpath.read_text()
    new_txt = re.sub(
        r"torch\.load\(([^,)]+)\)",
        r"torch.load(\1, weights_only=False)",
        txt,
    )
    if new_txt != txt:
        fpath.write_text(new_txt)
        print(f"  patched : {fpath.name}")

print("Patches OK")

  patched : loader_features.py
  patched : testing.py
  patched : embedding_projection.py
Patches OK


In [30]:
# ── Cell 5: MI-MM inference ──────────────────────────────────────────────────
# MI-MM uses S3D visual features + a sentence embedding text model with a
# multi-instance max-margin loss. testing.py is run from MI-MM/src/ so its
# default relative paths (../data/...) resolve through the symlink from Cell 2.
# Output is written by create_submission.py to MI-MM/output/test.pkl.

print("Running MI-MM inference ...")
r = subprocess.run(
    f'cd "{MIMM_DIR}/src" && python testing.py 2>&1',
    shell=True, capture_output=True, text=True, timeout=900,
)

for line in r.stdout.splitlines():
    if any(k in line for k in ["load", "Best", "Evaluat", "Error", "Traceback", "nDCG", "mAP"]):
        print(line)

if r.returncode != 0:
    print("\nFull output:", r.stdout[-1000:])
    print("STDERR:", r.stderr[-400:])
    raise RuntimeError("MI-MM inference failed")

mimm_raw = MIMM_OUTPUT_DIR / "test.pkl"
assert mimm_raw.exists(), f"Output not generated: {mimm_raw}"

with open(mimm_raw, "rb") as f:
    sub_mimm = pickle.load(f)

sim_mimm = np.array(sub_mimm["sim_mat"], dtype=np.float32)
assert sim_mimm.shape == (9668, 3842), f"Unexpected shape: {sim_mimm.shape}"

dst = SUBMISSIONS_DIR / "MI-MM_test_latest.pkl"
shutil.copy2(mimm_raw, dst)
print(f"\nMI-MM OK  sim_mat={sim_mimm.shape}  saved -> {dst.name}")

Running MI-MM inference ...
load features...
load model...
Best Epoch: 202
Evaluating...

MI-MM OK  sim_mat=(9668, 3842)  saved -> MI-MM_test_latest.pkl


In [31]:
# ── Cell 6: JPoSE inference ──────────────────────────────────────────────────
# JPoSE learns separate verb and noun embeddings (Joint Part-of-Speech) with
# a triplet loss, then concatenates them (comb-func=cat) at test time.
MODEL_NAME = "JPoSE_BEST"
COMB_FUNC  = "cat"

CHECKPOINT = MODELS_DIR / MODEL_NAME / "model" / f"EPIC_100_retrieval_{MODEL_NAME}.pth"
assert CHECKPOINT.exists(), f"Checkpoint not found: {CHECKPOINT}"

jpose_out = SUBMISSIONS_DIR / f"{MODEL_NAME}_test_latest.pkl"

print(f"Running JPoSE inference ({MODEL_NAME}, comb-func={COMB_FUNC}) ...")
r = subprocess.run(
    f'cd "{JPOSE_DIR}" && PYTHONPATH="{JPOSE_DIR}/src" '
    f'python -W ignore src/train/test_jpose_triplet.py "{CHECKPOINT}" '
    f'--comb-func {COMB_FUNC} --challenge-submission "{jpose_out}" --gpu True 2>&1',
    shell=True, capture_output=True, text=True, timeout=600,
)

print(r.stdout[-2000:])
if r.returncode != 0:
    print("STDERR:", r.stderr[-400:])
    raise RuntimeError("JPoSE inference failed")

assert jpose_out.exists(), f"Output not generated: {jpose_out}"
with open(jpose_out, "rb") as f:
    sub_jpose = pickle.load(f)

sim_jpose = np.array(sub_jpose["sim_mat"], dtype=np.float32)
assert sim_jpose.shape == (9668, 3842), f"Unexpected shape: {sim_jpose.shape}"
print(f"\nJPoSE OK  sim_mat={sim_jpose.shape}  saved -> {jpose_out.name}")

Running JPoSE inference (JPoSE_BEST, comb-func=cat) ...
--- Current Test Arguments ---
Namespace(batch_size=64, checkpoint_rate=10, embedding_size=256, gpu=True, learning_rate=0.01, margin=1.0, momentum=0.9, num_epochs=100, num_layers=2, optimiser='SGD', out_dir='./logs/runs', tt_weight=1.0, tv_weight=2.0, vt_weight=1.0, vv_weight=1.0, action_weight=1.0, comb_func='cat', comb_func_start=0, noun_weight=1.0, verb_weight=1.0, num_triplets=10, triplet_sampling_rate=10, online_hard=False)
nDCG: VT:0.707 TV:0.674 AVG:0.690
mAP: VT:0.757 TV:0.712 AVG:0.734


JPoSE OK  sim_mat=(9668, 3842)  saved -> JPoSE_BEST_test_latest.pkl


In [32]:
# ── Cell 7: MLP / MMEN inference ────────────────────────────────────────────
# MMEN (Multi-Modal Embedding Network) is the caption-only baseline from the
# JPoSE codebase. It uses only the caption text modality (no verb/noun split)
# with the same triplet loss. Weaker than JPoSE alone but adds complementary
# signal for the ensemble in Cell 8.
MMEN_MODEL = MODELS_DIR / "MMEN_BEST" / "model" / "EPIC_100_retrieval_MLP_BEST.pth"
assert MMEN_MODEL.exists(), f"Checkpoint not found: {MMEN_MODEL}"

mmen_out = SUBMISSIONS_DIR / "MMEN_BEST_test_latest.pkl"

print("Running MLP/MMEN inference ...")
r = subprocess.run(
    f'cd "{JPOSE_DIR}" && PYTHONPATH="{JPOSE_DIR}/src" '
    f'python -W ignore src/train/test_mmen_triplet.py "{MMEN_MODEL}" '
    f'--challenge-submission "{mmen_out}" --gpu True 2>&1',
    shell=True, capture_output=True, text=True, timeout=600,
)

print(r.stdout[-2000:])
if r.returncode != 0:
    print("STDERR:", r.stderr[-400:])
    raise RuntimeError("MLP/MMEN inference failed")

assert mmen_out.exists(), f"Output not generated: {mmen_out}"
with open(mmen_out, "rb") as f:
    sub_mmen = pickle.load(f)

sim_mlp = np.array(sub_mmen["sim_mat"], dtype=np.float32)
assert sim_mlp.shape == (9668, 3842), f"Unexpected shape: {sim_mlp.shape}"
print(f"\nMLP OK  sim_mat={sim_mlp.shape}  saved -> {mmen_out.name}")

Running MLP/MMEN inference ...
Namespace(batch_size=64, checkpoint_rate=10, embedding_size=256, gpu=False, learning_rate=0.01, margin=1.0, momentum=0.9, num_epochs=100, num_layers=2, optimiser='SGD', out_dir='./logs/runs', tt_weight=1.0, tv_weight=2.0, vt_weight=1.0, vv_weight=1.0, caption_type='caption', num_triplets=10, triplet_sampling_rate=10)
nDCG: VT:0.612 TV:0.577 AVG:0.595
mAP: VT:0.600 TV:0.540 AVG:0.570


MLP OK  sim_mat=(9668, 3842)  saved -> MMEN_BEST_test_latest.pkl


In [33]:
# ── Cell 8: Ensemble ─────────────────────────────────────────────────────────
# Each model uses a different visual backbone and loss function, so their errors
# are partially uncorrelated. Combining them recovers cases that any single
# model misses.
#
# Before summing we normalise each model's scores per-query (row) to [0, 1].
# Without this, models with wider score distributions dominate the blend
# regardless of quality.
#
# Weights: equal by default. To tune them, run the individual models against
# a validation split and set W_* proportional to their nDCG AVG scores.

def normalize_sim(sim):
    """Min-max normalise each row (query) to [0, 1]."""
    mn = sim.min(axis=1, keepdims=True)
    mx = sim.max(axis=1, keepdims=True)
    return (sim - mn) / (mx - mn + 1e-9)

# Sanity-check: all models must use the same ID ordering before adding scores.
assert list(sub_jpose["vis_ids"]) == list(sub_mimm["vis_ids"]),  "vis_ids mismatch MI-MM vs JPoSE"
assert list(sub_jpose["txt_ids"]) == list(sub_mimm["txt_ids"]),  "txt_ids mismatch MI-MM vs JPoSE"
assert list(sub_jpose["vis_ids"]) == list(sub_mmen["vis_ids"]),  "vis_ids mismatch MLP  vs JPoSE"
assert list(sub_jpose["txt_ids"]) == list(sub_mmen["txt_ids"]),  "txt_ids mismatch MLP  vs JPoSE"

W_MIMM, W_JPOSE, W_MLP = 1/3, 1/3, 1/3   # tune here if you have a val set

sim_ensemble = (
    W_MIMM  * normalize_sim(sim_mimm)  +
    W_JPOSE * normalize_sim(sim_jpose) +
    W_MLP   * normalize_sim(sim_mlp)
)

print(f"Ensemble shape  : {sim_ensemble.shape}")
print(f"Weights         : MI-MM={W_MIMM:.2f}  JPoSE={W_JPOSE:.2f}  MLP={W_MLP:.2f}")
print(f"Score range     : [{sim_ensemble.min():.4f}, {sim_ensemble.max():.4f}]")

Ensemble shape  : (9668, 3842)
Weights         : MI-MM=0.33  JPoSE=0.33  MLP=0.33
Score range     : [0.0077, 1.0000]


In [34]:
# ── Cell 9: Cross-modal manifold re-ranking ──────────────────────────────────
# Motivation: if two videos A and B are consistently ranked highly by the same
# text queries, they are likely semantically similar — even if the original
# model never explicitly compared them. We exploit this implicit structure to
# smooth scores along the data manifold.
#
# We approximate unimodal affinities through the cross-modal bridge:
#   A_vv[i,j] = cosine(row_i(S), row_j(S))   video-video affinity via text
#   A_tt[i,j] = cosine(col_i(S), col_j(S))   text-text  affinity via video
#
# Then run one step of diffusion on the similarity matrix:
#   S_new = (1-alpha)*S + alpha * 0.5*(A_vv @ S + S @ A_tt.T)
#
# The affinity matrices are sparsified to top-k neighbours (scipy sparse)
# before the diffusion step to keep memory and runtime manageable.
#
# Reference: Alpha-QE (Radenovic et al., ECCV 2018), adapted for cross-modal
# retrieval without requiring raw feature vectors.

from scipy.sparse import csr_matrix

def rerank(sim_mat, k=20, alpha=0.3):
    """
    Args:
        sim_mat : (N_vis, N_txt) float32 similarity matrix
        k       : neighbourhood size for expansion (higher = smoother)
        alpha   : expansion weight; 0 = unchanged, 1 = full diffusion
    Returns:
        Re-ranked (N_vis, N_txt) float32 similarity matrix.
    """
    S = sim_mat.astype(np.float64)

    # L2-normalise to compute cosine-based affinities
    S_v = S   / (np.linalg.norm(S,   axis=1, keepdims=True) + 1e-9)
    S_t = S.T / (np.linalg.norm(S.T, axis=1, keepdims=True) + 1e-9)
    A_vv = S_v @ S_v.T   # (N_vis, N_vis)
    A_tt = S_t @ S_t.T   # (N_txt, N_txt)

    # Sparsify: keep top-k per row, row-normalise, store as scipy sparse
    def to_sparse_topk(A, k):
        n    = A.shape[0]
        idx  = np.argpartition(A, -k, axis=1)[:, -k:]
        vals = np.take_along_axis(A, idx, axis=1)
        vals = vals / (vals.sum(axis=1, keepdims=True) + 1e-9)
        rows = np.repeat(np.arange(n), k)
        return csr_matrix((vals.ravel(), (rows, idx.ravel())), shape=(n, n))

    A_vv_sp = to_sparse_topk(A_vv, k)
    A_tt_sp = to_sparse_topk(A_tt, k)
    del A_vv, A_tt   # free ~750 MB

    # One-step diffusion.
    # For S @ A_tt.T: use (A_tt_sp @ S.T).T to keep the sparse term on the left.
    S_exp = 0.5 * (A_vv_sp @ S + (A_tt_sp @ S.T).T)
    return ((1 - alpha) * S + alpha * S_exp).astype(np.float32)


print("Re-ranking ensemble ...")
sim_reranked = rerank(sim_ensemble, k=20, alpha=0.3)

print(f"Re-ranked shape : {sim_reranked.shape}")
print(f"Parameters      : k=20  alpha=0.3")
print(f"Score range     : [{sim_reranked.min():.4f}, {sim_reranked.max():.4f}]")

Re-ranking ensemble ...
Re-ranked shape : (9668, 3842)
Parameters      : k=20  alpha=0.3
Score range     : [0.0317, 0.9922]


In [35]:
# ── Cell 10: Create submission ZIPs ──────────────────────────────────────────
# Serialise every variant as a Codabench-compatible ZIP so you can compare
# them on the leaderboard. Protocol 2 + the numpy._core patch ensures
# compatibility with the evaluator's Python / numpy version.

SLS_PT, SLS_TL, SLS_TD = 2, 3, 3

def make_compat_pickle(sim, vis_ids, txt_ids):
    payload = {
        "version":   "0.1",
        "challenge": "multi_instance_retrieval",
        "sls_pt": SLS_PT, "sls_tl": SLS_TL, "sls_td": SLS_TD,
        "sim_mat": np.array(sim, dtype=np.float32),
        "vis_ids": [str(v) for v in vis_ids],
        "txt_ids": [str(t) for t in txt_ids],
    }
    raw = pickle.dumps(payload, protocol=2)
    return raw.replace(b"numpy._core.multiarray", b"numpy.core.multiarray")

vis_ids = sub_jpose["vis_ids"]
txt_ids = sub_jpose["txt_ids"]
tmp_pkl = Path("/tmp/test.pkl")

# Listed best-expected first so you can upload from the top
submissions = [
    ("reranked",  sim_reranked),   # ensemble + cross-modal diffusion
    ("ensemble",  sim_ensemble),   # MI-MM + JPoSE + MLP normalised average
    ("JPoSE",     sim_jpose),      # best single model
    ("MI-MM",     sim_mimm),
    ("MLP",       sim_mlp),
]

print("Creating ZIPs:\n")
for name, sim in submissions:
    assert sim.shape == (9668, 3842)
    tmp_pkl.write_bytes(make_compat_pickle(sim, vis_ids, txt_ids))
    zip_path = ZIPS_DIR / f"{name}_submission.zip"
    subprocess.run(
        f'cd /tmp && zip -j "{zip_path}" test.pkl',
        shell=True, check=True, capture_output=True,
    )
    print(f"  {name:10s}  {zip_path.name}  ({zip_path.stat().st_size/1e6:.1f} MB)")

print(f"\nAll ZIPs saved to: {ZIPS_DIR}")
print("\nUpload to: https://www.codabench.org/competitions/12008")

Creating ZIPs:

  reranked    reranked_submission.zip  (154.0 MB)
  ensemble    ensemble_submission.zip  (153.8 MB)
  JPoSE       JPoSE_submission.zip  (155.6 MB)
  MI-MM       MI-MM_submission.zip  (165.5 MB)
  MLP         MLP_submission.zip  (164.8 MB)

All ZIPs saved to: /home/nico/Desktop/Multi-Instance-Retrieval-EK-100/EK100_MIR/submission_zips

Upload to: https://www.codabench.org/competitions/12008
